# BigQuery DataFrames: Enterprise Analytics from Prototyping to Production

This presentation notebook demonstrates the end-to-end enterprise data engineering workflow of BigQuery DataFrames. We will build, explore, and productionize a complex **Taxi Performance & Anomalies Pipeline** using a large public dataset.

### Core Features Showcased:
1. **Local Peek Cache**: Enables fast, interactive exploration on large datasets by caching intermediate results locally, avoiding costly full-table scans on every step.
2. **Python UDF Transpiler**: Compiles custom Python business logic (including multi-column row-wise functions and NumPy operations) directly into optimized SQL CASE expressions running on BigQuery's engine.
3. **Productionize Mode**: Wraps interactive data logic in a parameterized query pipeline, compiling it into an optimized SQL script or exporting it as a ready-to-run **Dataform** project directory.

## 1. Environment Setup & Option Configuration

We start by configuring the options to enable interactive caching and the Python UDF transpiler preview, then initialize our BigQuery DataFrames global session.

In [ ]:
import time
import bigframes.pandas as bpd
import numpy as np

# Configure to your BigQuery project
PROJECT_ID = "bigframes-dev"
bpd.options.bigquery.project = PROJECT_ID

# 1. Enable Local Peek Cache for fast exploration
bpd.options.compute.enable_peek_cache = True
bpd.options.compute.peek_cache_size = 5000

# 2. Enable Python UDF Transpiler for CASE-compilation
bpd.options.experiments.enable_python_transpiler = True

# Configure standard session parameters
bpd.options.bigquery.ordering_mode = 'partial'
bpd.options.display.render_mode = 'anywidget'

session = bpd.get_global_session()
print("BigQuery DataFrames session initialized!")

## 2. Phase 1: Interactive Prototyping & Exploration

We load the public Chicago Taxi Trips dataset containing over 200 million rows. We use a lazy reference, allowing us to build the logic without querying the full table.

In [ ]:
# Load the massive public dataset (200M+ rows)
df = bpd.read_gbq("bigquery-public-data.chicago_taxi_trips.taxi_trips")

df.peek(15)

In [ ]:
# Select a subset of relevant features
df_filtered = df[["payment_type", "company", "trip_seconds", "trip_miles", "fare", "tips", "trip_total"]].dropna()
#df_filtered['new_col'] = df_filtered.apply(lambda row: row.trip_miles/row.trip_seconds, axis=1)
df_filtered.peek(25)

### Applying Complex Multi-Column Python UDFs & NumPy Operations

Next, we want to perform complex business classification of trips (e.g., finding taxi speed anomalies or tip behaviors). We write a standard Python function and apply it to each row. The transpiler translates it automatically into SQL.

In [ ]:
# 1. Define custom classification function using multiple features
import math

def classify_trip_efficiency(row):
    # Calculate speed in mph (add 1 to avoid ZeroDivisionError)
    speed_mph = (row["trip_miles"] / (row["trip_seconds"] + 1)) * 3600.0
    
    # Calculate tip percentage
    tip_percent = (row["tips"] / (row["fare"] + 0.1)) * 100.0
    
    if speed_mph > 80.0:
        return "Anomaly (High Speed)"
    elif tip_percent > 20.0:
        return f"High Tip {math.ceil(tip_percent)}%"
    else:
        return "Standard"

# 2. Apply UDF and use NumPy operations for scaling/capping metrics
df_transformed = df_filtered.assign(
    trip_efficiency=df_filtered.apply(classify_trip_efficiency, axis=1),
    log_total=np.log1p(df_filtered["trip_total"]),
    clamped_fare=np.minimum(df_filtered["fare"], 100.0)
)

df_transformed.peek()

### Fast Iterative Analytics (Leveraging Local Peek Cache)

Now that the columns are added, we group and aggregate the dataset. With the peek cache enabled, subsequent operations on the aggregate result are near-instantaneous.

In [ ]:
# Group by company and trip efficiency category
df_company_metrics = df_transformed.groupby(["company", "trip_efficiency"]).agg(
    avg_fare=("fare", "mean"),
    avg_log_total=("log_total", "mean"),
    trip_count=("fare", "count")
).reset_index()

# Filter to find companies with relevant counts
df_explore = df_company_metrics[df_company_metrics["trip_count"] >= 500]

# Let's display the exploration df (cached locally after the first display/peek)
start_time = time.time()
df_explore_pd = df_explore.to_pandas()
print(f"First read time (querying BQ): {time.time() - start_time:.2f} seconds")

# Re-displaying/Sorting the dataframe is now instantaneous because of the Peek Cache
start_time = time.time()
df_explore_sorted = df_explore.sort_values(by="avg_fare", ascending=False).head(10).to_pandas()
print(f"Cached read/sort time (re-using peek cache): {time.time() - start_time:.4f} seconds")
df_explore_sorted

## 3. Phase 2: Scale & Productionize

Once we are happy with our prototype, we encapsulate it into a production-grade pipeline using `bpd.productionize()`. This mode lets us define parameters and record output tables.

In [ ]:
# Create dataset for production outputs
DATASET_ID = "bigframes_production_presentation"
session.bqclient.create_dataset(DATASET_ID, exists_ok=True)

with bpd.productionize() as pipeline:
    # 1. Define query parameter for minimum trip count dynamically
    min_trip_threshold = bpd.parameter("min_trip_threshold", dtype=int)
    
    # 2. Lazy data reference
    df_raw = bpd.read_gbq("bigquery-public-data.chicago_taxi_trips.taxi_trips")
    
    # 3. Filtering and feature selection
    df_sel = df_raw[["payment_type", "company", "trip_seconds", "trip_miles", "fare", "tips", "trip_total"]].dropna()
    
    # 4. Register custom classification as persistent BQ UDF
    def classify_trip_efficiency(row):
        # Calculate speed in mph (add 1 to avoid ZeroDivisionError)
        speed_mph = (row["trip_miles"] / (row["trip_seconds"] + 1)) * 3600.0
        
        # Calculate tip percentage
        tip_percent = (row["tips"] / (row["fare"] + 0.1)) * 100.0
        
        if speed_mph > 80.0:
            return "Anomaly (High Speed)"
        elif tip_percent > 20.0:
            return "High Tip"
        else:
            return "Standard"
    # 5. Apply the persistent UDF & numpy operations
    df_prod = df_sel.assign(
        trip_efficiency=df_sel.apply(
            classify_trip_efficiency,
            axis=1
        ),
        log_total=np.log1p(df_sel["trip_total"]),
        clamped_fare=np.minimum(df_sel["fare"], 100.0)
    )
    
    # 6. Perform Groupby and aggregation
    df_final = df_prod.groupby(["company", "trip_efficiency"]).agg(
        avg_fare=("fare", "mean"),
        avg_log_total=("log_total", "mean"),
        trip_count=("fare", "count")
    ).reset_index()

    # 7. Apply the threshold parameter
    df_filtered_final = df_final[df_final["trip_count"] >= min_trip_threshold]
    
    # 8. Define target table write
    df_filtered_final.to_gbq(
        f"{PROJECT_ID}.{DATASET_ID}.company_efficiency_report",
        if_exists="replace"
    )

print("Production pipeline successfully compiled!")

## 4. Phase 3: Developer & Enterprise Hand-off

### Compiled SQL Script

We can extract the full optimized compiled SQL script representing the entire pipeline. This script can be run directly inside BigQuery without python dependency.

In [ ]:
sql_script = pipeline.to_sql()
print("=== COMPILED SQL ===")
print(sql_script[:10000] + "\n... [TRUNCATED] ...")

### Exporting to Dataform

Alternatively, we can export the recorded pipeline directly into a local Dataform project. This directory can be committed directly to Git for production orchestrations (like weekly batch jobs).

In [ ]:
import tempfile
import os 

export_dir = os.path.join(tempfile.gettempdir(), "dataform_presentation")
os.makedirs(export_dir, exist_ok=True)

print(f"Exporting project code to Dataform in: {export_dir}")
pipeline.export_dataform(export_dir)

print("\nGenerated Dataform project files:")
for root, _, files in os.walk(export_dir):
    for file in files:
        rel_path = os.path.relpath(os.path.join(root, file), export_dir)
        print(f"- {rel_path}")